In [93]:
# pip install executorch
# uv add executorch

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import roi_align
from ultralytics import YOLO as UltralyticsYOLO
import timm
import lightning as L


torch.set_float32_matmul_precision('high')

CHARS = 'ABCDEFGHIJKLMNOPRSTUVWXYZ0123456789'
BLANK = len(CHARS)  # index 36 — blank / padding
NUM_CLASSES = len(CHARS) + 1
MAX_LEN = 8
CHAR2IDX = {c: i for i, c in enumerate(CHARS)}

model_name = 'model_4_synth_2.pt'


class PlateClassifier(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, max_len: int = MAX_LEN):
        super().__init__()
        self.backbone = timm.create_model(
            "mobilenetv3_small_100",
            pretrained=True,
            num_classes=1,
            drop_rate=0.2,
        )
        self.backbone.classifier = nn.Identity()
        self.dropout = nn.Dropout(0.15)
        self.head = nn.Linear(1024, max_len * num_classes)
        self.max_len = max_len
        self.num_classes = num_classes

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.head(x)
        return x.view(-1, self.max_len, self.num_classes)


IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}


def _decode(t):
    return "".join(IDX2CHAR.get(i.item(), "") for i in t if i.item() != BLANK)


class PlateModule(L.LightningModule):
    def __init__(self, lr: float = 1e-3, weight_decay: float = 1e-4, unfreeze_epoch: int = 3):
        super().__init__()
        self.model = PlateClassifier()

        _yolo_wrapper = UltralyticsYOLO('./yolo_best.pt')
        self.yolo = _yolo_wrapper.model.float()
        for m in self.yolo.modules():
            if hasattr(m, 'export'):
                m.export = True

        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        self.lr = lr
        self.weight_decay = weight_decay
        self.unfreeze_epoch = unfreeze_epoch
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        for p in self.model.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):                                    # [1, 3, 960, 960] float32 NCHW [0-255]
        det  = self.yolo(x / 255.0)                         # [1, 300, 6]  x1 y1 x2 y2 conf cls
        conf = det[:, :, 4:5]
        best_idx = conf.argmax(dim=1, keepdim=True)
        best_det = torch.gather(det, 1, best_idx.expand(-1, -1, 6)).squeeze(1)

        x1, y1 = best_det[:, 0:1], best_det[:, 1:2]
        x2, y2 = best_det[:, 2:3], best_det[:, 3:4]
        best_conf = best_det[:, 4:5]  # ← confidence

        # crop
        batch_idx = torch.zeros_like(x1)
        rois    = torch.cat([batch_idx, x1, y1, x2, y2]).unsqueeze(0)   # [1, 5]
        cropped = roi_align(x, rois, output_size=(64, 224), spatial_scale=1.0) # [1, 3, 64, 224]

        # OCR
        normalized = (cropped / 255.0 - self.mean) / self.std
        chars    = self.model(normalized)                    # [1, 8, 37]
        best_box = torch.cat([x1, y1, x2, y2, best_conf], dim=1)  # [1, 5] zamiast [1, 4]

        return best_box, chars

    def on_train_epoch_start(self):
        if self.current_epoch == self.unfreeze_epoch:
            for p in self.model.backbone.parameters():
                p.requires_grad = True

    def _step(self, batch):
        imgs, labels = batch
        logits = self.model(imgs)
        loss = self.loss_fn(logits.permute(0, 2, 1), labels)
        preds = logits.argmax(-1)
        char_acc  = (preds == labels).float().mean()
        plate_acc = (preds == labels).all(dim=-1).float().mean()
        return loss, char_acc, plate_acc


In [14]:
module = PlateModule(lr=1e-3)

model = PlateClassifier()  # mobilenetv3_small_100, num_features=1024
model.load_state_dict(torch.load(model_name, map_location='cpu'))

<All keys matched successfully>

In [15]:

# export friendly
class PlateModuleExportable(torch.nn.Module):
    def __init__(self, plate_module):
        super().__init__()
        self.yolo = plate_module.yolo
        self.model = plate_module.model
        self.register_buffer('mean', plate_module.mean)
        self.register_buffer('std', plate_module.std)

    def forward(self, x):
        det = self.yolo(x / 255.0)
        conf = det[:, :, 4:5]
        best_idx = conf.argmax(dim=1, keepdim=True)
        best_idx_expanded = best_idx.expand(-1, -1, 6)
        best_det = torch.gather(det, 1, best_idx_expanded).squeeze(1)  # [1, 6]

        x1, y1 = best_det[:, 0:1], best_det[:, 1:2]
        x2, y2 = best_det[:, 2:3], best_det[:, 3:4]
        best_conf = best_det[:, 4:5]  # ← confidence

        # Normalizuj koordynaty do [-1, 1] dla grid_sample
        h, w = x.shape[2], x.shape[3]
        x1_n = 2.0 * x1 / w - 1.0
        y1_n = 2.0 * y1 / h - 1.0
        x2_n = 2.0 * x2 / w - 1.0
        y2_n = 2.0 * y2 / h - 1.0

        # Siatka 224x224 w znormalizowanej przestrzeni
        grid_x = torch.linspace(0, 1, 224, device=x.device).view(1, 1, 224, 1)
        grid_y = torch.linspace(0, 1, 64, device=x.device).view(1, 64, 1, 1)

        sample_x = x1_n + (x2_n - x1_n) * grid_x  # [1, 1, 224, 1]
        sample_y = y1_n + (y2_n - y1_n) * grid_y  # [1, 64, 1, 1]

        grid = torch.cat([
            sample_x.expand(-1, 64, -1, -1),
            sample_y.expand(-1, -1, 224, -1),
        ], dim=-1)  # [1, 64, 224, 2]

        cropped = F.grid_sample(x, grid, mode='bilinear', align_corners=False)  # [1, 3, 64, 224]
        normalized = (cropped / 255.0 - self.mean) / self.std
        chars = self.model(normalized)
        best_box = torch.cat([x1, y1, x2, y2, best_conf], dim=1)

        return best_box, chars


exportable = PlateModuleExportable(module)
exportable.eval()

# Test
with torch.no_grad():
    box, chars = exportable(torch.randn(1, 3, 960, 960))
    print(f"box: {box.shape}, chars: {chars.shape}")

box: torch.Size([1, 5]), chars: torch.Size([1, 8, 36])


In [11]:
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms.functional as TF

CHARS = 'ABCDEFGHIJKLMNOPRSTUVWXYZ0123456789'

img = Image.open('dataset/plates_data/CWL/audi_a4_CWL93YT.jpg').resize((960, 960))
tensor = TF.to_tensor(img).unsqueeze(0) * 255

module = PlateModule()
module.model.load_state_dict(torch.load(model_name, map_location='cpu'))
module.eval()

# 1. PyTorch z roi_align
with torch.no_grad():
    _, chars_pt = module(tensor)
idx = chars_pt.argmax(-1)[0]
text = ''.join(CHARS[i] if i < len(CHARS) else '_' for i in idx)
print(f'roi_align:    {text}')

# 2. Exportable z grid_sample
exportable = PlateModuleExportable(module)
exportable.eval()
with torch.no_grad():
    _, chars_ex = exportable(tensor)
idx2 = chars_ex.argmax(-1)[0]
text2 = ''.join(CHARS[i] if i < len(CHARS) else '_' for i in idx2)
print(f'grid_sample:  {text2}')

roi_align:    CWL93YT_
grid_sample:  CWL93YT_


In [12]:
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.runtime import Runtime

CHARS = 'ABCDEFGHIJKLMNOPRSTUVWXYZ0123456789'

# model
module = PlateModule()
module.model.load_state_dict(torch.load(model_name, map_location='cpu'))
module.eval()

exportable = PlateModuleExportable(module)
exportable.eval()


img = Image.open('dataset/plates_data/CWL/audi_a4_CWL93YT.jpg').resize((960, 960))
tensor = TF.to_tensor(img).unsqueeze(0) * 255


with torch.no_grad():
    _, chars_pt = exportable(tensor)

# Export
exported = torch.export.export(exportable, (torch.randn(1, 3, 960, 960),), strict=False)
print("Export OK")


et_out = exported.module()(tensor)
idx = et_out[1].argmax(-1)[0]
text = ''.join(CHARS[i] if i < len(CHARS) else '_' for i in idx)
print(f'Po export (przed lowering): {text}')

# Lowering
program = to_edge_transform_and_lower(exported, partitioner=[XnnpackPartitioner()]).to_executorch()
print("Lowering OK")

with open("model_new.pte", "wb") as f:
    program.write_to_file(f)


runtime = Runtime.get()
method = runtime.load_program("model_new.pte").load_method("forward")
out = method.execute([tensor])
idx2 = out[1].argmax(-1)[0]
text2 = ''.join(CHARS[i] if i < len(CHARS) else '_' for i in idx2)
print(f'Po lowering (.pte):         {text2}')

diff = (chars_pt - out[1]).abs().max().item()
print(f'Max diff: {diff:.4f}')

Failed to load /home/oobx/Desktop/PROJEKTY/SIiUM/.venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/oobx/Desktop/PROJEKTY/SIiUM/.venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /home/oobx/Desktop/PROJEKTY/SIiUM/.venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/oobx/Desktop/PROJEKTY/SIiUM/.venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
W0623 22:51:11.606000 3336302 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Export OK
Po export (przed lowering): CWL93YT_


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Lowering OK
Po lowering (.pte):         CWL93YT_
Max diff: 0.0000


In [11]:
ckpt = torch.load(model_name, map_location='cpu')
state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt


print("Checkpoint keys (5 pierwszych):")
for k in list(state_dict.keys())[:5]:
    print(f"  {k}")

print("\nModel keys (5 pierwszych):")
for k in list(model.state_dict().keys())[:5]:
    print(f"  {k}")

Checkpoint keys (5 pierwszych):
  backbone.conv_stem.weight
  backbone.bn1.weight
  backbone.bn1.bias
  backbone.bn1.running_mean
  backbone.bn1.running_var

Model keys (5 pierwszych):
  backbone.conv_stem.weight
  backbone.bn1.weight
  backbone.bn1.bias
  backbone.bn1.running_mean
  backbone.bn1.running_var
